# P06-A — Generator data sintesis untuk latihan Bab 6

Notebook ini membuat tiga CSV latihan untuk Orange dan Python. Jalankan di Google Colab atau Jupyter. Data sepenuhnya sintetis; pola dihasilkan oleh rumus yang ditulis di bawah, bukan hasil pengujian laboratorium atau survei jalan. Seed tetap menjaga hasil dapat diulang.

In [ ]:
# Folder kerja: jalankan dari folder mana pun. Sel ini pindah ke akar repositori agar jalur data relatif (praktikum/...) benar.
import os
from pathlib import Path
_akar = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "praktikum").is_dir() and (p / "notebooks").is_dir()), None)
if _akar is not None:
    os.chdir(_akar)
else:
    print("Akar repositori tidak ditemukan; memakai folder kerja saat ini (normal di Colab).")

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 260616
rng = np.random.default_rng(SEED)
# Dari akar repositori: tulis ke praktikum/bab06/data_bab6; di Colab atau folder lain: data_bab6 di direktori kerja.
out = Path("praktikum/bab06/data_bab6") if Path("praktikum/bab06").is_dir() else Path("data_bab6")
out.mkdir(parents=True, exist_ok=True)
print("Folder keluaran:", out)

Folder keluaran: praktikum/bab06/data_bab6


## 1. Regresi stabilitas Marshall

Satu baris mewakili satu rancangan campuran unik. Respons dibangkitkan dari fungsi nonlinear sederhana, lalu diberi noise. Persamaan ini hanya perangkat pedagogis.

In [2]:
n = 800
binder = np.clip(rng.normal(5.1, 0.38, n), 4.0, 6.2)
va = rng.uniform(3.0, 7.0, n)
vma = rng.uniform(13.0, 21.0, n)
shape = rng.uniform(0.30, 0.95, n)
gradation = rng.uniform(2.2, 3.4, n)
stability = (24 + 5.5*(binder-5.1) - 0.85*(va-4.7)**2 + 0.62*(vma-16.5) + 3.2*(shape-0.62) + 1.1*(gradation-2.8) + rng.normal(0, 2.2, n))
marshall = pd.DataFrame({"mix_id":[f"M{i:04d}" for i in range(n)],"binder_pct":binder,"air_void_pct":va,"vma_pct":vma,"aggregate_shape_index":shape,"gradation_index":gradation,"stability_kN":stability})
marshall.to_csv(out/"marshall_regression.csv", index=False)
marshall.head()

,mix_id,binder_pct,air_void_pct,vma_pct,aggregate_shape_index,gradation_index,stability_kN
0,M0000,5.537416,4.064792,16.987231,0.638727,2.992074,26.288964
1,M0001,5.402115,3.160310,16.766820,0.913383,2.202521,23.146475
2,M0002,5.052117,4.626198,16.569923,0.847919,2.697116,26.577888
3,M0003,4.866293,6.785479,16.533745,0.507660,2.340077,14.586179
4,M0004,4.954183,4.528968,20.030956,0.678024,2.839621,26.679222


## 2. Klasifikasi prioritas segmen perkerasan

Label dibuat dari skor sintesis yang menggabungkan indikator kondisi. Ambang kuantil menghasilkan kelas prioritas yang lebih jarang. Karena label dibentuk dari fitur ini, latihan menguji pemulihan aturan buatan, bukan risiko lapangan.

In [3]:
n = 1200
iri = rng.uniform(0.7, 6.5, n)
crack = rng.beta(1.5, 5.0, n) * 35
rut = rng.gamma(2.0, 0.35, n)
deflection = rng.lognormal(-0.25, 0.45, n)
adt = rng.uniform(1.0, 35.0, n)
score = 0.34*(iri/6.5) + 0.27*(crack/35) + 0.18*(rut/2.5) + 0.16*(deflection/2.0) + 0.05*(adt/35)
cut = np.quantile(score, 0.83)
priority = np.where(score >= cut, "tinggi", "rendah")
road = pd.DataFrame({"segment_id":[f"S{i:04d}" for i in range(n)],"iri_m_per_km":iri,"crack_pct":crack,"rut_depth_cm":rut,"deflection_mm":deflection,"adt_thousand":adt,"priority":priority})
road.to_csv(out/"pavement_priority.csv", index=False)
road["priority"].value_counts(normalize=True)


priority
rendah    0.83
tinggi    0.17
Name: proportion, dtype: float64

## 3. Clustering profil kondisi jalan

Tiga rezim sintetis dipakai untuk membangkitkan fitur. `synthetic_reference` hanya untuk evaluasi eksternal setelah clustering selesai. Jangan masukkan kolom tersebut ke algoritma clustering.

In [4]:
n = 900
regime = rng.choice([0,1,2], size=n, p=[0.42,0.36,0.22])
means = np.array([[1.7, 4.0, 0.35, 0.45],[3.2, 12.0, 0.85, 0.85],[5.2, 25.0, 1.55, 1.45]])
sds = np.array([[0.45,2.5,0.18,0.20],[0.65,4.0,0.28,0.28],[0.75,6.0,0.35,0.35]])
x = np.vstack([rng.normal(means[g], sds[g]) for g in regime])
cluster = pd.DataFrame({"segment_id":[f"C{i:04d}" for i in range(n)],"iri_m_per_km":x[:,0].clip(0.5,8),"crack_pct":x[:,1].clip(0,60),"rut_depth_cm":x[:,2].clip(0,3),"deflection_mm":x[:,3].clip(0,3),"synthetic_reference":regime})
cluster.to_csv(out/"pavement_clustering.csv", index=False)
print("CSV dibuat:", *[p.name for p in out.glob("*.csv")], sep="\n- ")

CSV dibuat:
- pavement_priority.csv
- pavement_clustering.csv
- marshall_regression.csv


## Batas penggunaan

Generator ini memberi contoh proses yang dapat direproduksi dan dikritisi. Rumus, sebaran, dan label dibuat secara manual untuk latihan. Dataset tidak merepresentasikan survei, campuran standar, uji laboratorium, inventaris perkerasan, atau aturan prioritas lembaga. Sebelum memakai data lapangan, dokumentasikan satuan, protokol ukur, populasi, waktu, lokasi, dan izin penggunaan.

## Mengambil CSV di Google Colab
Berkas tetap berada pada lingkungan runtime Colab. Unduh setiap CSV yang diperlukan sebelum menutup atau mengganti runtime; unggah berkas tersebut pada notebook analisis bila notebook berjalan di runtime terpisah.

In [5]:
# Opsional: unduh CSV dari Google Colab ke komputer Anda.
try:
    from google.colab import files
    for path in out.glob("*.csv"):
        files.download(str(path))
except ImportError:
    print("Di Jupyter lokal, berkas berada di", out)


Di Jupyter lokal, berkas berada di praktikum/bab06/data_bab6
